# Manufacturing Multi-Agent — Vector Search Index for Service Bulletins

Creates a Vector Search endpoint and index on `service_bulletins.bulletin_text`
for the retrieval sub-agent to use.

**Databricks AI Features:** Vector Search, Embeddings, Unity Catalog


Install LangChain packages first. `%pip` restarts Python, so it must run before `_config` sets up widgets and `sys.path`.


In [ ]:
%pip install databricks-langchain langchain-core -q


## Setup
Loads catalog, schema, Vector Search endpoint, and embedding model from widgets.


In [ ]:
%run ./_config


Re-read job widgets in this notebook so catalog and schema are defined after `%run`.


In [ ]:
# Job parameters / widgets must be read in this notebook. %run does not always export names from .ipynb.
catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
try:
    llm_endpoint = dbutils.widgets.get("llm_endpoint").strip()
except Exception:
    llm_endpoint = "databricks-claude-sonnet-4-5"
try:
    embedding_endpoint = dbutils.widgets.get("embedding_endpoint").strip()
except Exception:
    embedding_endpoint = "databricks-gte-large-en"
try:
    vs_endpoint = dbutils.widgets.get("vs_endpoint").strip()
except Exception:
    vs_endpoint = "mfg_quality_vs_endpoint"
try:
    eval_sample_size = dbutils.widgets.get("eval_sample_size").strip() or "50"
except Exception:
    eval_sample_size = "50"
try:
    cost_owner = dbutils.widgets.get("cost_owner").strip()
except Exception:
    cost_owner = ""
VS_INDEX_NAME = f"{catalog}.{schema}.service_bulletins_index"
print(f"Bound widgets: {catalog}.{schema}")


Resolves source table and index names, then creates a workspace client.


In [ ]:
source_table = f"{catalog}.{schema}.service_bulletins"
VS_ENDPOINT_NAME = vs_endpoint
EMBEDDING_MODEL = embedding_endpoint

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.vectorsearch import (
    EndpointType,
    VectorIndexType,
    DeltaSyncVectorIndexSpecRequest,
    EmbeddingSourceColumn,
    PipelineType,
)
import time

w = WorkspaceClient()


## Step 1: Create Vector Search Endpoint (if not exists)


In [ ]:
# Check if endpoint exists
existing_endpoints = [e.name for e in w.vector_search_endpoints.list_endpoints()]
if VS_ENDPOINT_NAME not in existing_endpoints:
    print(f"Creating Vector Search endpoint: {VS_ENDPOINT_NAME}")
    w.vector_search_endpoints.create_endpoint(
        name=VS_ENDPOINT_NAME,
        endpoint_type=EndpointType.STANDARD,
    )
    # Wait for endpoint to be ready
    while True:
        ep = w.vector_search_endpoints.get_endpoint(VS_ENDPOINT_NAME)
        if ep.endpoint_status and ep.endpoint_status.state.value == "ONLINE":
            print(f"✅ Endpoint {VS_ENDPOINT_NAME} is ONLINE")
            break
        print(f"  Endpoint status: {ep.endpoint_status.state.value}... waiting 30s")
        time.sleep(30)
else:
    print(f"✅ Endpoint {VS_ENDPOINT_NAME} already exists")


## Step 2: Enable Change Data Feed on Source Table


In [ ]:
spark.sql(f"ALTER TABLE {source_table} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
print(f"✅ Change Data Feed enabled on {source_table}")


## Step 3: Create Delta Sync Vector Index


In [ ]:
# Check if index exists
existing_indexes = [idx.name for idx in w.vector_search_indexes.list_indexes(VS_ENDPOINT_NAME)]
if VS_INDEX_NAME not in existing_indexes:
    print(f"Creating Vector Search index: {VS_INDEX_NAME}")
    w.vector_search_indexes.create_index(
        name=VS_INDEX_NAME,
        endpoint_name=VS_ENDPOINT_NAME,
        primary_key="bulletin_id",
        index_type=VectorIndexType.DELTA_SYNC,
        delta_sync_index_spec=DeltaSyncVectorIndexSpecRequest(
            source_table=source_table,
            pipeline_type=PipelineType.TRIGGERED,
            embedding_source_columns=[
                EmbeddingSourceColumn(
                    name="bulletin_text",
                    embedding_model_endpoint_name=EMBEDDING_MODEL,
                )
            ],
        ),
    )
    print(f"  Index creation initiated. This may take a few minutes...")

    # Wait for index to be ready
    while True:
        idx = w.vector_search_indexes.get_index(VS_INDEX_NAME)
        status = idx.status
        if status and status.ready:
            print(f"✅ Index {VS_INDEX_NAME} is READY")
            break
        msg = status.message if status else "initializing"
        print(f"  Index status: {msg}... waiting 30s")
        time.sleep(30)
else:
    print(f"✅ Index {VS_INDEX_NAME} already exists")
    # Trigger a sync to pick up any new data
    w.vector_search_indexes.sync_index(VS_INDEX_NAME)
    print(f"  Sync triggered")


## Step 4: Test the Vector Index


In [ ]:
# Test query — find service bulletins related to brake rotor issues
test_query = "brake rotor thickness out of spec causing safety concern"

results = w.vector_search_indexes.query_index(
    index_name=VS_INDEX_NAME,
    columns=["bulletin_id", "part_type", "failure_mode", "severity_level", "bulletin_text"],
    query_text=test_query,
    num_results=3,
)

print(f"Query: '{test_query}'\n")
for i, row in enumerate(results.result.data_array):
    print(f"--- Result {i+1} (score: {row[-1]:.4f}) ---")
    print(f"  Bulletin: {row[0]}")
    print(f"  Part: {row[1]} | Failure: {row[2]} | Severity: {row[3]}")
    print(f"  Text preview: {row[4][:200]}...")
    print()


## Step 5: Test with LangChain Retriever


In [ ]:
from databricks_langchain import DatabricksVectorSearch

retriever = DatabricksVectorSearch(
    endpoint=VS_ENDPOINT_NAME,
    index_name=VS_INDEX_NAME,
    columns=["bulletin_id", "part_type", "failure_mode", "severity_level", "priority"],
).as_retriever(search_kwargs={"k": 3})

# Test retrieval
docs = retriever.invoke("wiring harness intermittent failure")
for doc in docs:
    print(f"Bulletin: {doc.metadata['bulletin_id']} | {doc.metadata['part_type']} - {doc.metadata['failure_mode']}")
    print(f"  {doc.page_content[:150]}...")
    print()

print(f"✅ Vector Search index is working. Ready for Sub-Agent 2 (Retriever).")
print(f"\nNext: Run 03_Sub_Agents.py")
